In [10]:
import pandas as pd
import os
import json
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord


In [14]:
prot_info_path = '../data/dbptm/dbptm_info.json'
splits_folder = os.path.dirname(prot_info_path)
prot_info = pd.read_json(prot_info_path)

In [3]:
prot_info.sequence[1]

'MYYFSRVAARTFCCCIFFCLATAYSRPDRNPRKIEKKDKKFFGASKNTNPANAMGNLFKAPTIEYVVEEVTRTHQPEQYDIPTDMSPLMTIAASESADKFTDKFFVDQSSIMKEKTSSKGNARTLL'

In [4]:
def get_prots(prot_info, splits_folder, residues = ['S', 'T', 'Y', 'ST', 'STY'], prefix=''):

    total_prots = set()
    for res in residues:
        with open(f'{splits_folder}/{prefix}splits_{res}.json', 'r') as f:
            splits = json.load(f)

        prots = set()
        for split in splits:
            for k, indices in split.items():
                prots.update(prot_info.loc[indices].id)

        total_prots.update(prots)

    return total_prots

In [15]:
total_prots = get_prots(prot_info, splits_folder)
filtered_prot_info = prot_info[prot_info['id'].apply(lambda x: x in total_prots)]
records = filtered_prot_info.apply(lambda x: SeqRecord(Seq(x['sequence']), id=x['id'], description=''), axis=1)
SeqIO.write(records, f'{splits_folder}/split_prots.fasta', 'fasta')

8546

In [49]:
filtered_prot_info.to_json(f'{prot_info_path.removesuffix('.json')}_split_prots_only.json', indent=2)

In [ ]:
paths = ['../data/uniptm/uniptm_info_S.json', '../data/uniptm/uniptm_info_T.json', '../data/uniptm/uniptm_info_Y.json', '../data/deeppsp/dpsp_info_ST.json', '../data/deeppsp/dpsp_info_Y.json']
residues = [['S'], ['T'], ['Y'], ['ST'], ['Y']]
prefixes = ['uniptm_', 'uniptm_', 'uniptm_', '', '']
splits_folders = [os.path.dirname(prot_info_path) for prot_info_path in paths]

for path, folder, res, prefix in zip(paths, splits_folders, residues, prefixes):
    prot_info = pd.read_json(path)
    total_prots = get_prots(prot_info, folder, residues=res, prefix=prefix)
    filtered_prot_info = prot_info[prot_info['id'].apply(lambda x: x in total_prots)]
    filtered_prot_info.to_json(f'{path.removesuffix('.json')}_split_prots_only.json', indent=2)

In [7]:
prot_info_path = '../data/phosphosite_sequences/phosphosite_df.json'
splits_folder = '../data/dataset_clusters_cov1_c05_c0_aln_3_s6/'
prot_info = pd.read_json(prot_info_path)

total_prots = get_prots(prot_info, splits_folder)
filtered_prot_info = prot_info[prot_info['id'].apply(lambda x: x in total_prots)]
filtered_prot_info.to_json(f'{splits_folder}/phosphosite_df_split_prots_only.json', indent=2)

In [11]:
records = filtered_prot_info.apply(lambda x: SeqRecord(Seq(x['sequence']), id=x['id'], description=''), axis=1)

In [13]:
SeqIO.write(records, f'{splits_folder}/split_prots.fasta', 'fasta')

7048